In [1]:
!uv add gitsource

Resolved 122 packages in 1.83s
Prepared 2 packages in 228ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 2 packages in 28ms
 + gitsource==0.0.5
 + python-frontmatter==1.3.0


In [40]:
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
import os

groq_client = OpenAI(api_key=os.environ.get("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",)

response = groq_client.responses.create(
    model="llama-3.3-70b-versatile",
    input='messages')

# Preparation

In [41]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [42]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [43]:
len(files), len(documents)

(72, 72)

In [44]:
from minsearch import Index
index=Index(text_fields= ['content'],keyword_fields=["filename"])
index.fit(documents)

In [45]:
ans=index.search("How does the agentic loop keep calling the model until it stops?")

In [46]:
ans[0]

{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry 

In [47]:
from Module_01.rag_helper_for_hm import RAGBase
assistant = RAGBase(
    index=index,
    llm_client=groq_client,
)

ans=assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [48]:
ans.usage

ResponseUsage(input_tokens=10412, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=456, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=10868)

In [49]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [50]:
len(chunks)

295

In [51]:
index.fit(chunks)

In [52]:
from Module_01.rag_helper_for_hm import RAGBase
assistant = RAGBase(
    index=index,
    llm_client=groq_client,
)

ans=assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [53]:
ans.usage

ResponseUsage(input_tokens=4790, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=607, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=5397)

In [59]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(query)

search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query to find relevant FAQ entries."
                }
            },
            "required": ["query"]
        }
    }
}

import json

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = assistant.search(**args)

    result_json = json.dumps(result, indent=2)

    return result_json

instructions = """
You're a course teaching assistant. 
Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
""".strip()

def agent(question, max_iterations=5):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ]

    for iteration in range(max_iterations):
        print(f"--- Iteration {iteration + 1} ---")
        
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            tools=[search_tool]
        )
        
        # Check if the model wants to call a function
        if response.choices[0].finish_reason == "tool_calls":
            # Process tool calls
            for tool_call in response.choices[0].message.tool_calls:
                print(f"Model decided to call function: {tool_call.function.name} with arguments: {tool_call.function.arguments}")
                
                # Parse arguments and call the function
                args = json.loads(tool_call.function.arguments)
                if tool_call.function.name == "search":
                    result = assistant.search(**args)
                    result_json = json.dumps(result, indent=2)
                
                # Add assistant message and tool result to messages
                messages.append({"role": "assistant", "content": "", "tool_calls": [tool_call]})
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result_json
                })
        else:
            # Model returned a final message
            final_answer = response.choices[0].message.content
            print(final_answer)
            return final_answer
    
    return None


In [60]:
response=agent("How does the agentic loop work, and how is it different from plain RAG?")

--- Iteration 1 ---


BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=search={"query": "agentic loop definition"}</function>\n'}}